# Session 2 — Climate Model: Carbon Cycle and Temperature Dynamics

This notebook implements a **reduced-form climate model** inspired by DICE-type frameworks.  
It tracks the dynamics of atmospheric carbon, ocean carbon, radiative forcing, and temperature.  
The time step is `Δ` years (annual or 5-year), starting at year `t0`.

---

## Inputs, states and derived quantities
- **E(t)** : Exogenous CO₂ emissions (GtCO₂ per year; input)  
- **M_AT(t)** : Carbon mass in the atmosphere (GtC)  
- **M_UP(t)** : Carbon mass in the upper ocean (GtC)  
- **M_LO(t)** : Carbon mass in the lower ocean (GtC)  
- **F(t)** : Radiative forcing (W/m²)  
- **T_AT(t)** : Atmospheric temperature anomaly (°C)  
- **T_LO(t)** : Lower-ocean temperature anomaly (°C)  

---

## Carbon Cycle Equations

$$
M_{AT}(t) = (1 - \Delta b_{12}) M_{AT}(t-1) 
+ \Delta b_{12} \frac{M_{AT0}}{M_{UP0}} M_{UP}(t-1) 
+ \Delta \xi E(t-1)
$$

$$
M_{UP}(t) = \Delta b_{12} M_{AT}(t-1) 
+ \left(1 - \Delta b_{12} \tfrac{M_{AT0}}{M_{UP0}} - \Delta b_{23}\right) M_{UP}(t-1) 
+ \Delta b_{23} \frac{M_{UP0}}{M_{LO0}} M_{LO}(t-1)
$$

$$
M_{LO}(t) = \Delta b_{23} M_{UP}(t-1) 
+ \left(1 - \Delta b_{23} \tfrac{M_{UP0}}{M_{LO0}}\right) M_{LO}(t-1)
$$

---

## Radiative Forcing

$$
F(t) = F_{2 \times CO_2} \cdot 
\frac{\ln\!\left(\tfrac{M_{AT}(t-1)}{M_{AT}^{\text{pre}}}\right)}{\ln(2)}
$$

where  
- $F_{2 \times CO_2}$ is the forcing from doubling CO₂ (≈ 3.36 W/m²),  
- $M_{AT}^{\text{pre}}$ is preindustrial atmospheric carbon mass.  

---

## Temperature Dynamics

$$
T_{AT}(t) = T_{AT}(t-1) 
+ \Delta c_1 \, F(t) 
- \Delta c_1 \, \frac{F_{2 \times CO_2}}{T_{2 \times CO_2}} \, T_{AT}(t-1) 
- \Delta c_1 c_3 \, \big( T_{AT}(t-1) - T_{LO}(t-1) \big), $$

$$
T_{LO}(t) = T_{LO}(t-1) 
+ \Delta c_4 \, \big( T_{AT}(t-1) - T_{LO}(t-1) \big).
$$

---

## Key Parameters
- $\Delta$: time step (years)  
- $F_{2 \times CO_2}$: forcing for CO₂ doubling  
- $T_{2 \times CO_2}$: equilibrium climate sensitivity (°C per doubling)  
- $b_{12}, b_{23}$: carbon transfer rates (atmosphere ↔ ocean)  
- $c_1, c_3, c_4$: heat transfer coefficients  
- $\xi$: conversion factor CO2 to C.

---

## Purpose
This notebook will:
1. Initialize the state variables at year `t0`.  
2. Simulate carbon and temperature dynamics over time.  
3. Explore the effect of emissions scenarios on climate trajectories.  

---


## Session 2 — Objectives and prerequisites

**Learning objectives**
- Connect historical greenhouse-gas data to the original `Climate_Models.py` carbon-cycle and temperature blocks.
- Check the model against observed temperature anomalies.
- Use SSP emissions to compare forward-looking temperature paths.

**Prerequisites.** Slides for Session 2; simple column selection with Pandas; basic Matplotlib; units GtCO₂ versus GtC.


## Technical setup

Run the notebook from this session directory. The original course module and data files are stored next to it.

**Convention.** The initial climate state is given. Emissions are prescribed annual inputs (not optimized controls): inputs in row t act on the transition from t to t+1.
The mathematical reference is **climate_model.pdf**. Use calendar-year alignment: annual emissions in row t act over the interval starting at t. The final date requested in update_path is included.


In [ ]:
# Install dependencies in the current notebook kernel before running imports.
%pip install -r requirements.txt


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import Climate_Models as CM

# A — Historical Climate Dynamics

In this section, we will work with **historical CO₂ emissions data** collected by *Our World in Data*.  
The dataset provides **annual CO₂ emissions by country** (measured in tonnes of CO₂) starting from the Industrial Revolution.


In [ ]:
# Read historical annual CO2 emissions.
df = pd.read_csv("https://ourworldindata.org/grapher/annual-co2-emissions-per-country.csv?v=1&csvType=full&useColumnShortNames=true", storage_options={'User-Agent': 'Our World In Data data fetch/1.0'})
df = df.rename(columns={'entity': 'Entity', 'code': 'Code', 'year': 'Year'})

In [ ]:
# Select World observations and order them by year.
df_world = df[df["Code"] == "OWID_WRL"].sort_values("Year")

# Convert tonnes of CO2 to GtCO2 per year.
world_co2 = df_world["emissions_total"] / 10**9
year_co2 = df_world["Year"]
emissions_by_year = df_world.set_index("Year")["emissions_total"] / 10**9
df_world.head()

### A-1) plot the paths of carbon emissions


In [ ]:
# Your code here
pass

### A-2) We would like to simulate climate dynamics

#### Initializing the Climate System

Before we can simulate climate dynamics, we must **set the model at its pre-industrial equilibrium**.  
This means starting the system in the state it was around the year 1750, before large-scale industrial CO₂ emissions began.

- **Carbon stocks** are initialized at their **pre-industrial levels**:  
  - $M_{AT0} = M_{AT}^{\text{pre}}$ (atmosphere)  
  - $M_{UP0} = M_{UP}^{\text{pre}}$ (upper ocean)  
  - $M_{LO0} = M_{LO}^{\text{pre}}$ (lower ocean)  

- **Temperature anomalies** are set to **zero** at the start:  
  - $T_{AT0} = 0$ (no atmospheric anomaly)  
  - $T_{LO0} = 0$ (no ocean anomaly)  

- The **time horizon** spans from 1749 to 2024 with annual steps.


In [ ]:
# Historical experiment: start at preindustrial equilibrium.
p = CM.Params()
p.Delta = 1
p.t0 = 1749
p.tT = 2024
p.nT = (p.tT - p.t0) // p.Delta + 1

p.M_AT0 = p.mat
p.M_UP0 = p.mup
p.M_LO0 = p.mlo
p.T_AT0 = 0
p.T_LO0 = 0
p.M_CH40 = p.mch4

# No emissions in this first experiment.
path = CM.init_states(p)
path = CM.update_path(path, p, 1750, 2024)
CM.mat_to_df(path, p)

**Exercise:**
1. What is the path of atmospheric carbon `M_AT` and temperature `T_AT`? Use a plot.
2. Quickly comment your outcome.


In [ ]:
# Your code
pass

Write your answer here.


### A-3) Feed the climate system with carbon emissions

**Goal**

We’ll (i) load historical CO₂ emissions, (ii) express them in GtCO₂/year (the module converts CO₂ to carbon internally), (iii) inject them into the simulation matrix `path[:, i_E]`, (iv) update the climate path, and (v) plot and comment.

---

**Exercise**

1. Replace the emissions in `path[:, p.i_E]` by `world_co2` aligned by year (already done above) and run:
   ```python
   CM.update_path(path_pollution, p, 1750, 2024)
   ```
2. Plot the result in term of carbon accumulation and global atmospheric temperature.


In [ ]:
# Start a fresh path, then fill its emissions column by calendar year.
path_pollution = CM.init_states(p)
# Use emissions_by_year.reindex(path_pollution[:, p.i_time], fill_value=0).
# Simulate 1750–2024 and plot the results.

### A-4) Plot and comment: global temperature anomalies

**Goal**

Compare observed global near-surface temperature anomalies with the model-simulated atmospheric temperature anomaly obtained from historical emissions. Discuss similarities and differences in terms of level, trend, and variability.

---

**Instructions**

1. **Line plot (comparison).**  
   Plot `world_temp` against `world_temp_yr` (observed series) together with `path_pollution[:, p.i_T_AT]` (simulated series under historical emissions).  
   Make sure both are aligned by calendar year.  
   Label axes, add a legend, title, and grid.

2. **Basic diagnostics.**  
   - Comment how well the simulated path reproduces the observed warming trend.  
   - Discuss differences in the early period vs. recent decades.  
   - Note that the observed series includes short-term variability (ENSO, volcanoes) not captured by the simple model.  
   - Reflect on whether the model underestimates or overestimates the magnitude of warming.


In [ ]:
# Read observed temperature anomalies.
temperatures_data = pd.read_csv("https://ourworldindata.org/grapher/temperature-anomaly.csv?v=1&csvType=full&useColumnShortNames=true", storage_options={'User-Agent': 'Our World In Data data fetch/1.0'})
temperatures_data = temperatures_data.rename(columns={'entity': 'Entity', 'code': 'Code', 'year': 'Year'})
temperatures_world = temperatures_data[temperatures_data["Code"] == "OWID_WRL"]
world_temp = temperatures_world["near_surface_temperature_anomaly"]
world_temp_yr = temperatures_world["Year"]

# Plot observed and simulated temperatures here.

Write your answer here.


### A-5) Methane cycles

**Goal**

Extend the climate model to account for methane (CH₄), which has a shorter lifetime than CO₂ but a stronger warming potential per molecule. Methane dynamics can be represented by a simple decay equation and its radiative forcing contribution.

---

**Methane cycle (one-box model)**

Methane accumulates through emissions and decays with a characteristic lifetime of 9 years in this course calibration:

$$
M_{CH4}(t) = m_{CH4} + (1 - \delta \Delta)(M_{CH4}(t-1)-m_{CH4}) + \Delta E_{CH4}(t-1)
$$

- $M_{CH4}(t)$: atmospheric methane concentration (ppb).  
- $E_{CH4}(t)$: methane emissions (in ppb equivalents).  
- $\delta = 1/9$: annual decay rate.  
- $\Delta$: time step (1 year).
- $m_{CH4}$: preindustrial level of methan. 

---

**Radiative forcing from methane**

Methane’s forcing is approximately proportional to the square root of its concentration:

$$
F_{CH4}(t) = \alpha_{CH4}\,\Big(\sqrt{M_{CH4}(t)} - \sqrt{M_{CH4,0}}\Big)
$$

- $M_{CH4,0} \approx 722$ ppb (pre-industrial baseline).  
- $\alpha_{CH4} \approx 0.036 \,\text{W/m}^2 / \sqrt{\text{ppb}}$.  

---

**Total forcing**

The climate system now responds to combined forcing:

$$
F(t) = F_{CO2}(t) + F_{CH4}(t) + F_{\text{ex}}(t)
$$

where $F_{\text{ex}}$ (set to zero) can capture other greenhouse gases or exogenous forcing.

---

---

**Questions**

- Compare Global temperature, atmospheric methan and carbon into a subplot `1*3`.
- Comment your results.

---


In [ ]:
# Download Methan data.
# Feed the model with methan data.
# Call CM.update_path, which already includes the methane stock and its forcing.

# Read global methane emissions.
df_ch4 = pd.read_csv("https://ourworldindata.org/grapher/ghg-emissions-by-gas.csv?v=1&csvType=full&useColumnShortNames=true", storage_options={'User-Agent': 'Our World In Data data fetch/1.0'})
df_ch4 = df_ch4.rename(columns={'entity': 'Entity', 'code': 'Code', 'year': 'Year'})
df_ch4_world = df_ch4[df_ch4["Entity"] == "World"].sort_values("Year")

# Course conversion: tonnes CO2-equivalent -> Mt CH4 -> ppb per year.
methane_by_year = df_ch4_world.set_index("Year")["annual_emissions_ch4_total_co2eq"] / (10**6 * 29.8 * 2.78)
path_ch4_co2 = path_pollution.copy()

# Fill the methane input by year, simulate and plot the three panels.

Write your answer here.


# B — Forecasting temperatures based on SSP scenarios

Use the five **Shared Socioeconomic Pathways (SSPs)** as alternative CO₂ emissions inputs.

Start all five simulations in **2015**, with the same DICE initial carbon stocks and temperatures supplied by `CM.Params()`. Do not reuse the final states of the historical experiment in Part A.

The local CSV already contains **annual** global emissions in **GtCO₂/year**, from **2020 to 2100**. Use the observed global emissions for **2015–2019**, identically in every scenario, then the SSP column from 2020 onwards. No extrapolation or new interpolation is needed.

- SSP1: Sustainability
- SSP2: Middle of the road
- SSP3: Regional rivalry
- SSP4: Inequality
- SSP5: Fossil-fueled development

These are baseline emissions paths, not optimized climate policies. The historical bridge uses the same CO₂ series as Part A; it is a teaching convention, not a recalibration of DICE.

Source supplied with the course: [Our World in Data — IPCC scenarios explorer](https://ourworldindata.org/explorers/ipcc-scenarios).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load
df = pd.read_csv("Notebook_ClimateModels_SSP_data.csv",sep=";",decimal=",")

# Pick year + SSP columns
year = df["Year"]

# Plot
plt.figure()
plt.plot(year, df["SSP1"], label="SSP1",color="#1B9E77")
plt.plot(year, df["SSP2"], label="SSP2",color="#D95F02")
plt.plot(year, df["SSP3"], label="SSP3",color="#8C6D31")
plt.plot(year, df["SSP4"], label="SSP4",color="#2CA25F")
plt.plot(year, df["SSP5"], label="SSP5",color="#6F42C1")
plt.grid(True, alpha=0.3)
plt.legend(title="Scenario", ncol=2, frameon=False)
plt.title("Global CO₂ emissions — SSP baselines")
plt.xlabel("Year"); plt.ylabel("GtCO₂/year")
plt.tight_layout();
plt.show()

### B-1) Simulate temperature paths under SSP scenarios

1. Set an annual grid from **2015 to 2100**. Keep the DICE initial states from a fresh `CM.Params()`: atmospheric carbon 851 GtC, upper-ocean carbon 460 GtC, deep-ocean carbon 1740 GtC, atmospheric temperature 0.85 °C and deep-ocean temperature 0.007 °C.
2. Create five separate paths. Put the historical emissions for 2015–2019 in their first five rows and the corresponding SSP emissions for 2020–2100 in the remaining rows.
3. Simulate **2016 to 2100**: the 2015 row is the given initial state; emissions in 2015 drive the first transition to 2016.
4. Plot the five temperature paths and carbon stocks. Display the temperatures in 2100 and interpret the differences.

Write the SSP1, SSP2, SSP3, SSP4 and SSP5 calculations separately, following the same simple steps. Use explicit assignments and plotting commands.

**Methane convention.** Keep the module's initial concentration (1860 ppb) in all five paths. This file supplies CO₂ only: additional methane emissions are set to zero, so methane relaxes towards its preindustrial level identically in every scenario. These are comparisons within the course model, not complete IPCC projections.


In [ ]:
# A NEW calibration: do not copy the preindustrial states from Part A.
p2 = CM.Params()
p2.Delta = 1
p2.t0 = 2015
p2.tT = 2100
p2.nT = (p2.tT - p2.t0) // p2.Delta + 1

# Keep the DICE initial carbon stocks and temperatures from CM.Params().
path_SSP = CM.init_states(p2)

# The SSP file begins in 2020. Historical data fill 2015–2019.
historical_2015_2019 = emissions_by_year.loc[2015:2019]
ssp = df.set_index("Year").loc[2020:2100]

# Same historical bridge and methane assumption in every scenario.
path_SSP[:5, p2.i_E] = historical_2015_2019
path_SSP[:, p2.i_E_CH4] = 0

CM.mat_to_df(path_SSP, p2).head(1)

# Create path_SSP1 as a copy of path_SSP.
# Fill path_SSP1[5:, p2.i_E] with ssp["SSP1"].
# Run CM.update_path(path_SSP1, p2, 2016, 2100).
# Write the same three steps explicitly for SSP2, SSP3, SSP4 and SSP5.
# Plot the five results and display temperatures in 2100.

Write your answer here.
